In [0]:
CATALOG = "datamodeling"

checks = [
    # (nazwa testu, zapytanie zwracające liczbę błędnych wierszy)
    ("silver: brak duplikatów order_id", f"""
        SELECT COUNT(*) FROM (
            SELECT order_id FROM {CATALOG}.silver.silver_table
            GROUP BY order_id HAVING COUNT(*) > 1)"""),

    ("fact: brak duplikatów order_id", f"""
        SELECT COUNT(*) FROM (
            SELECT order_id FROM {CATALOG}.gold.fact_sales
            GROUP BY order_id HAVING COUNT(*) > 1)"""),

    ("fact: brak NULL w kluczach wymiarów", f"""
        SELECT COUNT(*) FROM {CATALOG}.gold.fact_sales
        WHERE customer_sk IS NULL OR product_sk IS NULL OR payment_sk IS NULL
           OR region_sk IS NULL OR date_key IS NULL"""),

    ("liczba wierszy: silver = fact", f"""
        SELECT ABS((SELECT COUNT(*) FROM {CATALOG}.silver.silver_table)
                 - (SELECT COUNT(*) FROM {CATALOG}.gold.fact_sales))"""),

    # dodatkowe, tanie i sensowne
    ("dimcustomers: unikalny customer_id", f"""
        SELECT COUNT(*) FROM (
            SELECT customer_id FROM {CATALOG}.gold.dimcustomers
            GROUP BY customer_id HAVING COUNT(*) > 1)"""),

    ("fact: sales_amount = quantity * unit_price", f"""
        SELECT COUNT(*) FROM {CATALOG}.gold.fact_sales
        WHERE sales_amount <> quantity * unit_price"""),
]

In [0]:
from datetime import datetime

results = []
for name, sql in checks:
    bad = spark.sql(sql).collect()[0][0]
    results.append((name, int(bad), "PASS" if bad == 0 else "FAIL", datetime.now()))

df = spark.createDataFrame(results, "check_name STRING, failed_rows INT, status STRING, run_ts TIMESTAMP")
display(df)

# historia testów - przydatne do monitoringu i do pokazania w README
df.write.mode("append").saveAsTable(f"{CATALOG}.gold.dq_results")

failed = [r for r in results if r[2] == "FAIL"]
if failed:
    raise Exception(f"Data quality failed: {[r[0] for r in failed]}")